#### We are given a table with titles of recipes from a cookbook and their page numbers. Our task is to produce a table that represents how the recipes are distributed across the pages of the cookbook. Specifically, for each even-numbered page (the left page), show the title of that page in one column, and in the next column, show the title of the next odd-numbered page (the right page).

- Each row should contain:
- left_page_number: The page number for the left side (even page).
- left_title: The title of the recipe on the left page.
- right_title: The title of the recipe on the adjacent right page.

If a page does not contain a recipe, the title should be NULL. Page 0 is guaranteed to be empty, so it will not appear in the result.

#### Solution
- Filtering for Left and Right Pages:
- We divide the original cookbook_titles DataFrame into two separate DataFrames based on whether the page number is even (left) or odd (right).
- We perform a left join between left_pages_df and right_pages_df. The condition for the join is that the left_page_number is exactly one less than the right_page_number (i.e., the left page is followed by the right page).
- We select the columns desired columns from the joined dataframe

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

In [0]:
spark = SparkSession.builder.appName("CookbookTitles").getOrCreate()

In [0]:
titles_data = [
    (1, 'Scrambled eggs'),
    (2, 'Fondue'),
    (3, 'Sandwich'),
    (4, 'Tomato soup'),
    (6, 'Liver'),
    (11, 'Fried duck'),
    (12, 'Boiled duck'),
    (15, 'Baked chicken')
]

In [0]:
titles_columns = ["page_number", "title"]

In [0]:
titles_df = spark.createDataFrame(titles_data, titles_columns)

In [0]:
left_pages_df = titles_df.filter(col("page_number") % 2 == 0).select("page_number", "title")
right_pages_df = titles_df.filter(col("page_number") % 2 != 0).select("page_number", "title")

In [0]:
left_pages_df = left_pages_df.withColumnRenamed("page_number", "left_page_number").withColumnRenamed("title", "left_title")
right_pages_df = right_pages_df.withColumnRenamed("page_number", "right_page_number").withColumnRenamed("title", "right_title")


In [0]:
result_df = left_pages_df.join(right_pages_df, left_pages_df.left_page_number + 1 == right_pages_df.right_page_number, "left").select("left_page_number", "left_title", "right_title")

In [0]:
result_df.show()

+----------------+-----------+-----------+
|left_page_number| left_title|right_title|
+----------------+-----------+-----------+
|               2|     Fondue|   Sandwich|
|               4|Tomato soup|       NULL|
|               6|      Liver|       NULL|
|              12|Boiled duck|       NULL|
+----------------+-----------+-----------+

